In [0]:
%run "../SetUp/setup" 

### Access Azure Data Lake using Service Principal
**Steps to follow:**
1. Register Azure AD Application/ Service Principal
2. Generate a secret/ password for the application
3. Set spark config with App/ Client Id, Directory/ Tenant Id & Secret
4. Assign role "Storage Blob Data Contributor" to the Data Lake

In [0]:
%run "../Includes/configs" 

In [0]:
%run "../Includes/comm_func" 

In [0]:
dbutils.widgets.text("p_file_date", "")

path,name,size,modificationTime
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-21/,2021-03-21/,0,1768733801000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-03-28/,2021-03-28/,0,1768733591000
abfss://raw@forrmulaa1dl.dfs.core.windows.net/2021-04-18/,2021-04-18/,0,1768733721000


[FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/__unitystorage/', name='__unitystorage/', size=0, modificationTime=1769227742000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta/', name='drivers_convert_to_delta/', size=0, modificationTime=1769359802000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/drivers_convert_to_delta_new/', name='drivers_convert_to_delta_new/', size=0, modificationTime=1769360049000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_external/', name='results_external/', size=0, modificationTime=1769228062000),
 FileInfo(path='abfss://demo@forrmulaa1dl.dfs.core.windows.net/results_partitioned/', name='results_partitioned/', size=0, modificationTime=1769228706000)]

In [0]:
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS f1_presentation.calculated_race_results(
  race_year INT,
  team_name STRING,
  driver_id INT,
  driver_name STRING,
  race_id INT,
  position INT,
  points INT,
  calculated_points INT,
  created_date TIMESTAMP,
  updated_date TIMESTAMP
)
USING DELTA

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW race_results_updated AS
SELECT *
FROM (
  SELECT
    races.race_year,
    constructors.name AS team_name,
    drivers.driver_id,
    drivers.name AS driver_name,
    races.race_id,
    results.position,
    results.points,
    11 - results.position AS calculated_points,
    ROW_NUMBER() OVER (
      PARTITION BY drivers.driver_id, races.race_id
      ORDER BY results.file_date DESC
    ) AS rn
  FROM f1_processed.results results
  JOIN f1_processed.drivers drivers
    ON results.driver_id = drivers.driver_id
  JOIN f1_processed.constructors constructors
    ON results.constructor_id = constructors.constructor_id
  JOIN f1_processed.races races
    ON results.race_id = races.race_id
  WHERE results.position <= 10
    AND results.file_date = '{v_file_date}'
)
WHERE rn = 1;

In [0]:
%sql
MERGE INTO f1_presentation.calculated_race_results tgt
USING race_results_updated upd
ON tgt.driver_id = upd.driver_id
AND tgt.race_id = upd.race_id

WHEN MATCHED THEN
  UPDATE SET
    tgt.position = upd.position,
    tgt.points = upd.points,
    tgt.calculated_points = upd.calculated_points,
    tgt.updated_date = current_timestamp

WHEN NOT MATCHED THEN
  INSERT (
    race_year, team_name, driver_id, driver_name,
    race_id, position, points, calculated_points, created_date
  )
  VALUES (
    upd.race_year, upd.team_name, upd.driver_id, upd.driver_name,
    upd.race_id, upd.position, upd.points, upd.calculated_points,
    current_timestamp
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
SELECT COUNT(1) FROM race_results_updated;

count(1)
0


In [0]:
%sql
SELECT COUNT(1) FROM f1_presentation.calculated_race_results;

count(1)
10055


In [0]:
%sql
--When we do a CTAS (create table as) syntax like this, the table is being created when you have selected the data, but when you do an incremental load, you are not going to create the table everytime. You would expect to have the table up front, so that you can merge the data into the table. So that means you either insert or update the records. 
--CREATE TABLE f1_presentation.calculated_race_results
--USING delta
--AS
--SELECT races.race_year, constructors.name AS team_name, drivers.name AS driver_name, results.position, results.points, 11 - results.position AS calculated_points FROM f1_processed.results 
--JOIN f1_processed.drivers ON (results.driver_id = drivers.driver_id)
--JOIN f1_processed.constructors ON (results.constructor_id = constructors.constructor_id)
--JOIN f1_processed.races ON (results.race_id = races.race_id)
--WHERE results.position <= 10;

num_affected_rows,num_inserted_rows
